<a href="https://colab.research.google.com/github/AnisaIntania/Natural-Language-Processing/blob/main/NLP_P5_08_Duckling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [26]:
!rm -rf /usr/local/envs/duckling
!mamba create -n duckling python=3.6 openjdk=8 -y
!/usr/local/envs/duckling/bin/python --version

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
[+] 0.0s
[+] 0.0s


Transaction

  Prefix: /usr/local/envs/duckling

  Updating specs:

   - python=3.6
   - openjdk=8


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + ca-certificates   2026.7.22  hbd8a1cb_0            conda-forge     Cached
  + ld_impl_linux-64     2.46.1  default_hbd61a6d_103  conda-forge     Cached
  + libffi                3.4.6  h2dba641_1            conda-forge       57kB
  + libgcc               16.2.0  ha9f2e26_7            conda-forge     Cached
  + libgcc-ng            16.2.0  h69a702a_7            conda-forge       29kB
  + libgomp      

In [27]:
!/usr/local/envs/duckling/bin/pip install JPype1==0.7.4 duckling==1.8.0

     |████████████████████████████████| 2.9 MB 4.8 MB/s            
     |████████████████████████████████| 4.2 MB 57.2 MB/s            
     |████████████████████████████████| 229 kB 52.4 MB/s            


In [28]:
!/usr/local/envs/duckling/bin/python -c "import jpype; from duckling import DucklingWrapper; print('OK', jpype.__version__)"

OK 0.7.4


In [31]:
import subprocess, json, atexit
from pprint import pprint

server_code = '''
import sys, json, os, glob
import jpype

try:
    jpype.getDefaultJVMPath()
except Exception:
    hits = glob.glob(os.path.join(sys.prefix, "**", "libjvm.so"), recursive=True)
    if hits:
        _p = hits[0]
        jpype.getDefaultJVMPath = lambda: _p

from duckling import DucklingWrapper
d = DucklingWrapper()

def send(o):
    sys.stdout.write("@@" + json.dumps(o, default=str) + "\\n")
    sys.stdout.flush()

send({"ready": True})
for line in sys.stdin:
    req = json.loads(line)
    try:
        send({"ok": getattr(d, req["m"])(*req["a"], **req["k"])})
    except Exception as e:
        send({"err": repr(e)})
'''
with open("duck_server.py", "w") as f:
    f.write(server_code)


class DucklingWrapper:
    def __init__(self):
        self._p = subprocess.Popen(
            ["/usr/local/envs/duckling/bin/python", "-u", "duck_server.py"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, universal_newlines=True, bufsize=1)
        self._read()
        atexit.register(self.close)

    def _read(self):
        log = []
        while True:
            line = self._p.stdout.readline()
            if not line:
                raise RuntimeError("Server Duckling berhenti:\n" + "".join(log))
            if line.startswith("@@"):
                return json.loads(line[2:])
            log.append(line)

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        def call(*args, **kwargs):
            self._p.stdin.write(json.dumps({"m": name, "a": args, "k": kwargs}) + "\n")
            self._p.stdin.flush()
            r = self._read()
            if "err" in r:
                raise RuntimeError(r["err"])
            return r["ok"]
        return call

    def close(self):
        try:
            self._p.kill()
        except Exception:
            pass

print("DucklingWrapper siap dipakai")

DucklingWrapper siap dipakai


In [32]:
d = DucklingWrapper()
pprint(d.parse_time(u"Let's meet at 11:45am"))
pprint(d.parse_time(u"Please call me today"))

[{'dim': 'time',
  'end': 21,
  'start': 11,
  'text': 'at 11:45am',
  'value': {'grain': 'minute',
            'others': [{'grain': 'minute', 'value': '2026-10-13T11:45:00.000Z'},
                       {'grain': 'minute', 'value': '2026-10-14T11:45:00.000Z'},
                       {'grain': 'minute',
                        'value': '2026-10-15T11:45:00.000Z'}],
            'value': '2026-10-13T11:45:00.000Z'}}]
[{'dim': 'time',
  'end': 20,
  'start': 15,
  'text': 'today',
  'value': {'grain': 'day',
            'others': [{'grain': 'day', 'value': '2026-10-12T00:00:00.000Z'}],
            'value': '2026-10-12T00:00:00.000Z'}}]


In [33]:
pprint(d.parse_time(u'Let\'s meet at 11:45am'))
pprint(d.parse_time(u'You owe me twenty bucks, please call me today'))

[{'dim': 'time',
  'end': 21,
  'start': 11,
  'text': 'at 11:45am',
  'value': {'grain': 'minute',
            'others': [{'grain': 'minute', 'value': '2026-10-13T11:45:00.000Z'},
                       {'grain': 'minute', 'value': '2026-10-14T11:45:00.000Z'},
                       {'grain': 'minute',
                        'value': '2026-10-15T11:45:00.000Z'}],
            'value': '2026-10-13T11:45:00.000Z'}}]
[{'dim': 'time',
  'end': 45,
  'start': 40,
  'text': 'today',
  'value': {'grain': 'day',
            'others': [{'grain': 'day', 'value': '2026-10-12T00:00:00.000Z'}],
            'value': '2026-10-12T00:00:00.000Z'}},
 {'dim': 'time',
  'end': 17,
  'start': 11,
  'text': 'twenty',
  'value': {'grain': 'year',
            'others': [],
            'value': '2020-01-01T00:00:00.000Z'}}]


In [34]:
pprint(d.parse_temperature(u'Let\'s change the temperatur from thirty two celsius to 65 degrees'))
pprint(d.parse_temperature(u"It's getting hotter day by day, yesterday it was thirty-five degrees celcius today its 37 degrees "))

[{'dim': 'temperature',
  'end': 65,
  'start': 55,
  'text': '65 degrees',
  'value': {'unit': 'degree', 'value': 65.0}},
 {'dim': 'temperature',
  'end': 51,
  'start': 33,
  'text': 'thirty two celsius',
  'value': {'unit': 'celsius', 'value': 32.0}}]
[{'dim': 'temperature',
  'end': 97,
  'start': 87,
  'text': '37 degrees',
  'value': {'unit': 'degree', 'value': 37.0}},
 {'dim': 'temperature',
  'end': 76,
  'start': 49,
  'text': 'thirty-five degrees celcius',
  'value': {'unit': 'celsius', 'value': 35.0}}]


In [35]:
pprint(d.parse_timezone(u"Let's meet at 10pm IST"))
pprint(d.parse_timezone(u"Let's meet at 22:00 EST"))

[{'dim': 'timezone',
  'end': 22,
  'start': 19,
  'text': 'IST',
  'value': {'value': 'IST'}}]
[{'dim': 'timezone',
  'end': 23,
  'start': 20,
  'text': 'EST',
  'value': {'value': 'EST'}}]


In [36]:
d.parse_number(u"Hey i am a 20 year old student from Alaska")

[{'dim': 'number',
  'text': '20',
  'start': 11,
  'end': 13,
  'value': {'value': 20.0}}]

In [37]:
d.parse_ordinal(u"I came 2nd and u came 1st in a race")

[{'dim': 'ordinal',
  'text': '2nd',
  'start': 7,
  'end': 10,
  'value': {'value': 2}},
 {'dim': 'ordinal',
  'text': '1st',
  'start': 22,
  'end': 25,
  'value': {'value': 1}}]

In [38]:
d.parse_money(u"This meal costs 3$")

[{'dim': 'amount-of-money',
  'text': '3$',
  'start': 16,
  'end': 18,
  'value': {'value': 3.0, 'unit': '$'}}]

In [39]:
d.parse_email(u"my email is abcxyz@gmail.com")

[{'dim': 'email',
  'text': 'abcxyz@gmail.com',
  'start': 12,
  'end': 28,
  'value': {'value': 'abcxyz@gmail.com'}}]

In [40]:
d.parse_duration(u"I have been working on this project for 4 hrs every month for almost 2 years.")

[{'dim': 'duration',
  'text': '4 hrs',
  'start': 40,
  'end': 45,
  'value': {'value': 4.0,
   'unit': 'hour',
   'year': None,
   'month': None,
   'day': None,
   'hour': 4,
   'minute': None,
   'second': None}},
 {'dim': 'duration',
  'text': '2 years',
  'start': 69,
  'end': 76,
  'value': {'value': 2.0,
   'unit': 'year',
   'year': 2,
   'month': None,
   'day': None,
   'hour': None,
   'minute': None,
   'second': None}}]

In [41]:
d.parse_url(u"The official website for the book Practical NLP is http://www.practicalnlp.ai/")

[{'dim': 'url',
  'text': 'http://www.practicalnlp.ai/',
  'start': 51,
  'end': 78,
  'value': {'value': 'http://www.practicalnlp.ai/'}},
 {'dim': 'url',
  'text': 'http://www.practicalnlp.ai/',
  'start': 51,
  'end': 78,
  'value': {'value': 'http://www.practicalnlp.ai/'}}]

In [42]:
d.parse_phone_number(u"Enter text here")#didnt demo this due to privacy reasons

[]